<!-- # <span style="color:red">UNDER CONSTRUCTION!!!!</span> -->

# Spoken Language Processing - Instituto Superior Técnico
### Laboratory Assignment 2 - Automatic Age Estimation Challenge
<!--[image](imgs/lab2_slp_banner.png)-->
<img src="imgs/lab2_slp_banner.png" alt="drawing" width="400"/>

# WEEK 2 - Using pre-trained models


During this week, students will implement two modern systems for age regression based on:
- speaker representations (utterance-based) obtained with an x-vector model (this notebook);
- speech representations (frame-based) obtained with a self-supervised learning (SSL) pre-trained model (`lab2_ssl.ipynb` notebook).

In both cases, students are encouraged to explore different feature configurations and alternative downstream models.

## Before starting

Let's import some modules and make some definitions. 

Like in the previous Notebooks, you need to upload pf_tools.py and requirements.txt if you are working on Google Colab. Otherwise, you should skip or delete the following code cell:

In [ ]:
!pip install -r requirements.txt # Run this cell if you are using Google colab

**WARNING from professors** We changed the pf_tools.py script for this second week. Be sure to update (the new one is compatible with Week 1 lab)

In [ ]:
import os
import csv
import pickle
import numpy as np
import librosa
import torch

from pf_tools import CheckThisCell, SLPdata
from speechbrain.inference.classifiers import EncoderClassifier
from speechbrain.utils.data_utils import split_path
from sklearn.svm import LinearSVC, SVR
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt


GENDER_CLASSES = ('F',  'M')
GEN2ID = {'F':0, 'M':1}
ID2GEN = dict((GEN2ID[k],k)for k in GEN2ID)

Like in the previous Notebooks, you need to mount Google drive if you are working on Google Colab. Otherwise, you should skip or delete the following code cell:

In [ ]:

raise CheckThisCell ## <---- Remove this torun this cell if you are on Google Colab
from google.colab import drive
drive.mount('/content/drive')


Like in week1, the audio data is expected to be in a folder with the following format:

```
lab2_data/
├── train/
│   └── wav/
│       └──wav files
│   └── info.csv
│
└── train_small/
    └── wav/
        └──wav files
    └── info.csv
...
```

You must already have this from the previous week, so you can set-up your data directory:

In [ ]:

raise CheckThisCell ## <---- Remove this after completing/checking this cell

CWD = os.getcwd()
DATADIR = f'{CWD}/lab2_data/' # <--- Change this variable to your working directory containig the SLP data
if not os.path.isdir(DATADIR):
    os.mkdir(DATADIR)
    print(f'WARNING: Your data is not in the folder {DATADIR}')

os.chdir(CWD)
print(f'Your SLP data should be in this folder {DATADIR}')


If you need to download again the data, you can run the following cell:

In [ ]:
raise CheckThisCell

os.chdir(DATADIR)

# download train
!wget http://groups.tecnico.ulisboa.pt/speechproc/pf26/lab2/train.tgz
!tar -xzvf train.tgz

#download train100
!wget http://groups.tecnico.ulisboa.pt/speechproc/pf26/lab2/train_small.tgz
!tar -xzvf train_small.tgz

#download dev
!wget http://groups.tecnico.ulisboa.pt/speechproc/pf26/lab2/dev.tgz
!tar -xzvf dev.tgz

#download evl
!wget http://groups.tecnico.ulisboa.pt/speechproc/pf26/lab2/evl.tgz
!tar -xzvf evl.tgz

os.chdir(CWD)

## Using pre-trained speaker embeddings (x-vectors)

The goal of this part of the lab is to become familiar with and show how to use pre-trained speaker embedings (a.k.a. x-vectors) for speech classification/regressions tasks.

There exist plenty of resources and pre-trained models that can be  useful for our task. In particular, x-vectors are the current state of the art approach to obtain speech embeddings that characterize very efficiently speaker or language, among others. X-vectors are neural models typically trained for speaker identification in a supervised way, but also in some cases for other related tasks. Once trained, they can be used to obtain a single embedding vector of fixed dimension for each audio input. This vector corresponds to the activations of one of the layers after the pooling layer.

The following are examples of x-vector models available in the `speechbrain` module:

- `speechbrain/spkrec-xvect-voxceleb`: same with a different architecture: https://huggingface.co/speechbrain/spkrec-xvect-voxceleb

- `speechbrain/spkrec-ecapa-voxceleb`: trained using a large speaker corpus for speaker verification: https://huggingface.co/speechbrain/spkrec-ecapa-voxceleb



The following code cell shows how to import one of those models to obtain an embedding vector:

In [ ]:
raise CheckThisCell ## <---- Remove this after completing/checking this cell 

# Instantiate the model. If you don't have GPU available, run this line
xvector_model = EncoderClassifier.from_hparams(source="speechbrain/spkrec-xvect-voxceleb", savedir=f"{CWD}/tmp")

# If you have GPU available, run this line instead
# xvector_model = EncoderClassifier.from_hparams(source="speechbrain/spkrec-xvect-voxceleb", savedir=f"{CWD}/tmp", run_opts={"device":"cuda"})

signal = xvector_model.load_audio(f'{DATADIR}/train_small/wav/00834c0e904d40eda496e55010acebc5.wav')
emb =  xvector_model.encode_batch(signal)

print(type(emb), emb.shape)

These (very informative) embedding vectors can be used to train simple models for several speech classification tasks, achieving excellent results. In particular, in this lab assignment, we will train a simple Support Vector Regression (SVR) on top of these x-vectors.


Student groups will be graded depending on their ability to explore different feature configurations and model alternatives/configurations.

### 1. Extracting x-vectors for the SLP datasets

Just like in Part1, we will code the feature transformation to process all data and obtain x-vectors. In this case, the function should receive as arguments the audio filename and an instance of `EncoderClassifier` (the x-vector model) and return the numpy array with the features. You must complete the following code using the previous example:

In [ ]:
raise CheckThisCell ## <---- Remove this after completing/checking this cell

# This function receives a filename and one encoder model
# (for instance, xvector_model object in previous cell example) and returns
# the extracted x-vectors as a numpy array of shape (1, embedding_dimension).
# Notice that the encoder extractor module generates a softlink to each audio
# file. To avoid generating "junk", you can delete this link also in the
# function

def extract_xvec(filename, emb_model):
    # LABWORK :: CODE TO INSERT HERE

    # LABWORK :: CODE TO INSERT HERE

    # Remove the "annoying" soft-link
    _, fl = split_path(filename)
    if os.path.islink(fl):
        os.remove(fl)
    return embedding

# This must return a numpy array of shape (1,D). 
# D can vary depending on the model you are using, but it is usually 192 or 512.
emb = extract_xvec(f'{DATADIR}/train_small/wav/00834c0e904d40eda496e55010acebc5.wav' , xvector_model)

# Check that this shape and type
# it should be: (1, 512) <class 'numpy.ndarray'> or (1, 192) <class 'numpy.ndarray'>
print(emb.shape, type(emb))

Let's generate the x-vectors for all our data sets using the SLP class and store in disk. Like in Part 1, we can keep different transformation configurations in a dictionary for later usage.  

Let's define first our configurations (you can try to different x-vector models, the propsed ones or even other that you may find in huggingface):

In [ ]:

raise CheckThisCell ## <---- Remove this after completing/checking this cell

transform = {
                'spkrec-ecapa-voxceleb' : # <--- You chan look for different models in speechbrain
                {
                    'audio_transform':
                        lambda x : extract_xvec(x,
                            emb_model = EncoderClassifier.from_hparams(
                                        source="speechbrain/spkrec-ecapa-voxceleb", # <--- You can look for different models in speechbrain
                                        savedir=f"{CWD}/tmp/spkrec-ecapa-voxceleb"
                                        )
                            ), ## <--- You need to modify this here
                    'chunk_transform': None,
                    'chunk_size': 0,
                    'chunk_hop':0
                }
            }

transform['spkrec-xvect-voxceleb'] = {
                    'audio_transform':
                        lambda x : extract_xvec(x,
                            emb_model = EncoderClassifier.from_hparams(
                                        source="speechbrain/spkrec-xvect-voxceleb", # <--- You can look for different models in speechbrain
                                        savedir=f"{CWD}/tmp/spkrec-xvect-voxceleb"
                                        )
                            ), ## <--- You need to modify this here
                    'chunk_transform': None,
                    'chunk_size': 0,
                    'chunk_hop':0
                }


And now let's do feature extraction. Be patient because this process can be a bit slow depending on the resources of your machine (train_small without GPU should take around 5min on google colab):

In [ ]:

raise CheckThisCell ## <---- Remove this after completing/checking this cell

# Download and feature extract
trainset = 'train_small'
transform_id = 'spkrec-xvect-voxceleb'

slp_partitions = {}
# for partition in ('train', 'train_small', 'dev', 'evl'):
for partition in ('train_small', 'dev', 'evl'):
    slp_partitions[partition] = SLPdata(DATADIR, partition,
                    transform_id=transform_id,
                    audio_transform=transform[transform_id]['audio_transform'],
                    chunk_transform=transform[transform_id]['chunk_transform'],
                    chunk_size=transform[transform_id]['chunk_size'],
                    chunk_hop=transform[transform_id]['chunk_hop']
                    )


### 2. Training an SVR model

Our first attempt of age regression system based on x-vectors will be a simple SVR model like in the `openSMILE` baseline, but in this case we will be using x-vectors as features.

First, we will use the SLP data instances to store the x-vectors, the labels and file identifiers in numpy arrays:

In [ ]:
from pf_tools import prepare_slp_data

#   Concatenate all data and labels
#   Each row corresponds to a file
#   We store the data, labels and file identifiers of each partition in dictionaries
#     with the partition name as key


gender_label_pos = 0
age_label_pos = 1

data, labels_gender, labels_age, fileids = {}, {}, {}, {}
# for partition in ('train', 'train_small', 'dev', 'evl'):
for partition in ('train_small', 'dev', 'evl'):
    data_and_labels = prepare_slp_data(slp_partitions[partition])
    print(f'Partition: {partition}')
    print(f'Number of samples: {data_and_labels["data"].shape[0]}')
    print(f'Number of features: {data_and_labels["data"].shape[1]}')
    print(f'Number of labels: {len(np.unique(data_and_labels["label"][:,gender_label_pos]))}')
    print(f'Number of identifiers (samples): {len(np.unique(data_and_labels["identifiers"]))}')
    print('---')
    data[partition] = data_and_labels['data']
    labels_gender[partition] = data_and_labels['label'][:,gender_label_pos]
    labels_age[partition] = data_and_labels['label'][:,age_label_pos]
    fileids[partition] = data_and_labels['identifiers']



Now, we will use `sklearn` Support Vector Regression (SVR) to:
1. Train our regressor and save it for later use.
2. Predict on the dev and evl partitions and save the results

In [ ]:
from sklearn.svm import SVR
from pf_tools import save_model

trainset = 'train_small'

# Train a SVR
model = SVR(kernel='linear') ### <---- a linear SVR
model.fit(data[trainset], labels_age[trainset])  ## <---- train MODEL

model_id = save_model(model, f'svr_{transform_id}', f'{DATADIR}/{trainset}/models/')
print(f'Model {model_id} saved in {DATADIR}/{trainset}/models/')

# Predict the dev and evl sets
dev_results = model.predict(data['dev']) #  Predict dev
filename = f'{DATADIR}/{trainset}/models/{model_id}/dev.pkl'
pickle.dump({'hyp':dev_results, 'fileids':fileids['dev']}, open(filename, 'wb'))

evl_results = model.predict(data['evl']) #  Predict evl
filename = f'{DATADIR}/{trainset}/models/{model_id}/evl.pkl'
pickle.dump({'hyp':evl_results, 'fileids':fileids['evl']}, open(filename, 'wb'))


It should be extremely easy to experiment other models provided in the `sklearn` module, including SVMs with other kernels, Random Forests, etc.


#### 2.1 Analyze results on the dev set and prepare your submission file

Let's check our performance on the dev set:

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

ref, hyp = labels_age['dev'], dev_results

print(f'Mean Absolute Error: {mean_absolute_error(ref, hyp):.2f}')
print(f'Mean Squared Error: {mean_squared_error(ref, hyp):.2f}')

You should obtain a mean absolute error around 7.6 (it will depend on the xvector model chosen). 

Now, let's generate the final prediction file and make a submission to the  [Kaggle competition](https://www.kaggle.com/t/8d80747e0c474688a83024aabdfe1ab0):

In [ ]:
from pf_tools import create_submission_file

students_group = '00' # <--- CHANGE THIS ACCORDINGLY

model_id = 'svr_spkrec-xvect-voxceleb_2026-05-14_01:46:15'
model_id_short = 'svr_spkrec-xvect'

results_path = f'{DATADIR}/{trainset}/models/{model_id}/'
filename = f'{CWD}/g{students_group}_{trainset}_{model_id_short}.csv' # <--- CHANGE THIS ACCORDINGLY

create_submission_file(results_path, filename)


At this point, you can explore different x-vector model configurations for feature extraction and alternative models to the SVR.

### 3. Training a neural network model

As an alternative to the SVR, we will explore simple neural models on top of x-vector features.

We will need to define the size of the feature vector that will be used as input to the neural network:

In [ ]:
raise CheckThisCell ## <---- Remove this after completing/checking this cell
feat_dim = 512 ### This depends on the xvector model you are using 

And a simple neural model architecture (you can change this):

In [ ]:
import torch
from torch import nn

# Define a simple linear neural model
class LinearRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(LinearRegressor, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        return self.model(x)

model = LinearRegressor(input_dim=feat_dim, hidden_dim=50)

We will use a simple `train_nn` function included in the `pf_tools` script that will permit training the model using backpropagation. Students are encouraged to explore this function and, eventually, to modify it to experiment alternative training strategies, parameters, etc.

In [ ]:
from pf_tools import train_nn, predict_nn, save_model

train_nn(model, slp_partitions[trainset], slp_partitions['dev'], batch_size=16, epochs=1000, lr=0.001,)

model_id = save_model(model, f'nnet_{transform_id}', f'{DATADIR}/{trainset}/models/')
print(f'Model {model_id} saved in {DATADIR}/{trainset}/models/')

#### 3.1 Analyze results on the dev set and prepare your submission file

Let's check our performance on the dev set:

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Predict the dev set
hyp, ref, files = predict_nn(model, slp_partitions['dev'])
filename = f'{DATADIR}/{trainset}/models/{model_id}/dev.pkl'
pickle.dump({'hyp':hyp, 'fileids':files}, open(filename, 'wb'))

# Report the results
print(f'Mean Absolute Error: {mean_absolute_error(ref, hyp):.2f}')
print(f'Mean Squared Error: {mean_squared_error(ref, hyp):.2f}')

# Predict the evl set
hyp, ref, files = predict_nn(model, slp_partitions['evl'])
filename = f'{DATADIR}/{trainset}/models/{model_id}/evl.pkl'
pickle.dump({'hyp':hyp, 'fileids':files}, open(filename, 'wb'))

And generate the final prediction file and make a submission to the  [Kaggle competition](https://www.kaggle.com/t/8d80747e0c474688a83024aabdfe1ab0):

In [ ]:
from pf_tools import create_submission_file

students_group = '00' # <--- CHANGE THIS ACCORDINGLY

# model_id = 'nnet_spkrec-xvect-voxceleb_2026-05-14_02:05:38'
model_id_short = 'nnet_spkrec-xvect-voxceleb'

results_path = f'{DATADIR}/{trainset}/models/{model_id}/'
filename = f'{CWD}/g{students_group}_{trainset}_{model_id_short}.csv' # <--- CHANGE THIS ACCORDINGLY

create_submission_file(results_path, filename)

At this point, you can explore different x-vector model configurations for feature extraction and alternative  neural model architectures and parameters.

# Contacts and support
You can contact the professors during the classes or the office hours.

Particularly, for this second laboratory assignment, you should contact Prof. Alberto Abad: alberto.abad@tecnico.ulisboa.pt


